In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. Load toàn bộ dữ liệu
df = pd.read_csv('GiveMeSomeCredit-training.csv').iloc[:, 1:] # Bỏ cột ID đầu tiên
target = 'SeriousDlqin2yrs'

# 2. Xử lý giá trị thiếu (Data Imputation)
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

# 3. Feature Engineering 
# Tạo biến tỷ lệ sử dụng tín dụng trên mỗi dòng nợ (tăng khả năng phân loại)
df['utilization_per_line'] = df['RevolvingUtilizationOfUnsecuredLines'] / (df['NumberOfOpenCreditLinesAndLoans'] + 1)

# Log Transform cho các biến có dải giá trị cực rộng (Skewed)
df['log_MonthlyIncome'] = np.log1p(df['MonthlyIncome'])
df['log_DebtRatio'] = np.log1p(df['DebtRatio'])

# 4. Xử lý Outliers bằng Clipping (Percentile 1% và 99%)
# Ngăn chặn các giá trị dị biệt làm hỏng trọng số mô hình
cols_to_limit = ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio', 'MonthlyIncome']
for col in cols_to_limit:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

# Đảo nhãn gốc (Gốc 1: Nợ xấu -> 0: Không nên cho vay | Gốc 0: Tốt -> 1: Nên cho vay)
df[target] = 1 - df[target]

# Đổi tên cột mục tiêu thành Chấp thuận/Nên cho vay
df = df.rename(columns={target: 'approved'})
# ==================================================================

# 5. Xuất toàn bộ dữ liệu sạch
df.to_csv('gmsc.csv', index=False)
print(f"Thành công: Đã xử lý và lưu toàn bộ {len(df)} dòng vào gmsc.csv")

Thành công: Đã xử lý và lưu toàn bộ 150000 dòng vào gmsc.csv
